# Binary membrane vs. soluble — ESM-2 650M

Predict whether a protein is **membrane-bound** or **soluble** from its sequence — the other
standard DeepLoc task, where protein language models reach the low-90s. Fine-tunes ESM-2 650M;
proteins with unknown membrane annotation (U) are excluded per the benchmark.

**Runtime:** Colab **T4 GPU**. ~15–25 min (much shorter than the 10-class run).

## 1 · Setup

In [ ]:
!pip -q install -U transformers datasets scikit-learn accelerate

In [ ]:
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), 'Enable T4: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

## 2 · Load the binary M/S split

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

raw = load_dataset('proteinea/deeploc')
def frame(split):
    d = raw[split].to_pandas()[['input','membrane']].rename(columns={'input':'sequence','membrane':'m'})
    d = d[d['m'].isin(['M','S'])].dropna().drop_duplicates('sequence')
    return d
train_full, test = frame('train'), frame('test')
train, val = train_test_split(train_full, test_size=0.1, stratify=train_full['m'], random_state=42)
print(f'{len(train)} train / {len(val)} val / {len(test)} test')
print(train['m'].value_counts().to_dict())

## 3 · Tokenize (dual-end truncation)

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
MODEL_NAME='facebook/esm2_t33_650M_UR50D'; MAX_LENGTH=1024
lab2id={'M':0,'S':1}; NAMES={'M':'Membrane-bound','S':'Soluble'}
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
def dual_end(s,n):
    if len(s)<=n: return s
    h=n//2; return s[:h]+s[-(n-h):]
def to_ds(df):
    ds=Dataset.from_pandas(df.reset_index(drop=True))
    def f(b):
        t=tok([dual_end(s,MAX_LENGTH-2) for s in b['sequence']], truncation=True, max_length=MAX_LENGTH)
        t['labels']=[lab2id[x] for x in b['m']]; return t
    return ds.map(f, batched=True, remove_columns=['sequence','m'])
train_ds,val_ds,test_ds=to_ds(train),to_ds(val),to_ds(test)
collator=DataCollatorWithPadding(tok)

## 4 · Fine-tune

If you hit CUDA OOM, set `per_device_train_batch_size=1`, `gradient_accumulation_steps=32`.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0:'M',1:'S'}, label2id=lab2id)
model.gradient_checkpointing_enable(); model.config.use_cache=False

def metrics(p):
    pred=np.argmax(p.predictions,axis=-1)
    return {'accuracy':accuracy_score(p.label_ids,pred),'macro_f1':f1_score(p.label_ids,pred,average='macro'),'mcc':matthews_corrcoef(p.label_ids,pred)}

args=TrainingArguments(output_dir='out_bin', num_train_epochs=3,
    per_device_train_batch_size=2, per_device_eval_batch_size=4, gradient_accumulation_steps=8,
    learning_rate=2e-5, lr_scheduler_type='cosine', warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='accuracy', greater_is_better=True, save_total_limit=1,
    fp16=True, logging_steps=20, report_to='none')
trainer=Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=metrics, data_collator=collator)
trainer.train()

## 5 · Evaluate & export for the dashboard

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import json
pred=trainer.predict(test_ds); y_true,y_pred=pred.label_ids,np.argmax(pred.predictions,axis=-1)
acc=accuracy_score(y_true,y_pred)
print('TEST accuracy %.4f | macro-F1 %.4f | MCC %.4f' % (acc, f1_score(y_true,y_pred,average='macro'), matthews_corrcoef(y_true,y_pred)))
print(classification_report(y_true,y_pred,target_names=['Membrane-bound','Soluble']))

pr,rc,f1c,sup=precision_recall_fscore_support(y_true,y_pred,labels=[0,1],zero_division=0)
cm=confusion_matrix(y_true,y_pred,labels=[0,1])
names=['Membrane-bound','Soluble']
result={'task':'Membrane-bound vs. soluble','source':'fine-tuned ESM-2 650M',
  'accuracy':float(acc),'macro_f1':float(f1_score(y_true,y_pred,average='macro')),
  'mcc':float(matthews_corrcoef(y_true,y_pred)),'n_test':int(len(y_true)),'classes':names,
  'per_class':[{'label':names[i],'precision':float(pr[i]),'recall':float(rc[i]),'f1':float(f1c[i]),'support':int(sup[i])} for i in range(2)],
  'confusion':cm.astype(int).tolist()}
open('binary_metrics.json','w').write(json.dumps(result,indent=2))
from google.colab import files; files.download('binary_metrics.json')

---
Model: ESM-2 650M (Lin et al., 2023) · Data: DeepLoc (Almagro Armenteros et al., 2017)